In [ ]:
# Q1: 학생 성적 보고서 생성기
import csv
import json 
import logging 
import sys
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)

def make_report(csv_path: str, json_path: str)-> int:
    #CSV 파일을 열어 학생들의 성적 데이터를 읽음, logging 모듈로 예외처리
    try:
        with open(csv_path,"r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            ss = 0 
            results = []

            for row in reader:
                if row["중간"] == "" or row["기말"] == ""or row["과제"] == "":
                    avg = None
                    grade = None
                else:
                    avg = int(row["중간"]) * 0.3 + int(row["기말"]) * 0.5 + int(row["과제"]) * 0.2
                    grade = "A" if avg >= 90 else "B" if avg >= 80 else "C" if avg >= 70 else "F"
                    ss += 1

                row['평균']= avg
                row['등급']= grade

                if avg is None:
                    logging.info(f"{row['이름']}: 결측값 있음, 평균 None, 등급 None")
                else:
                    logging.info(f"{row['이름']}: 평균 {avg}, 등급 {grade}")
                
                results.append({
                    "이름": row["이름"],
                    "학번": row["학번"],
                    "점수": {
                        "중간": int(row["중간"]) if row["중간"] != "" else None, "기말": int(row["기말"]) if row["기말"] != "" else None, "과제": int(row["과제"]) if row["과제"] != "" else None
                    },
                    "평균": avg,
                    "등급": grade
                })

    except FileNotFoundError:
        logging.warning("CSV 파일이 존재하지 않습니다")
        return 0
    except UnicodeDecodeError:
        logging.error("CSV 파일의 인코딩이 잘못되었습니다")
        return 0

    
    #학생들의 성적 데이터를 json 파일로 저장    
    with open(json_path,"w",encoding="utf-8") as f:
        json.dump(results,f,ensure_ascii=False, indent=2)

    return ss 

make_report("scores.csv","report.json")

        


INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 결측값 있음, 평균 None, 등급 None


3

설명: with 문을 이용하여, csv 파일을 열고, json파일을 저장할 때, 파일을 자동으로 닫아주게 하였다. 또한 한글 텍스트의 표준인 utf-8을 명시했으며, FileNotFound와 UnicodeDecodeError를 잡아 학생들 데이터 처리에 필요한 csv 파일이 없거나, 인코딩이 잘못되었을 시 사용자에게 알려주도록 하였다. 또한 logging.info을 출력할 때 <avg(평균)이 none이라면> if문을 사용하여 결측값인 경우에도 그 사실을 알 수 있도록 하였다. 그런데 logging.info가 gihub codespace에서 사용자에게 보이게 출력되지 않아서 sys 모듈을 추가하여, logging 출력이 셀 출력창에 표시되어 보이도록 했다. 
참조한 생성형 ai 링크: https://claude.ai/share/b3da44a4-c249-4692-af92-2fb71a7f45ba

(1) 생성된 report.json의 전체 내용:
[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]
(2) 화면에 출력되는 logging 메시지:
INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 결측값 있음, 평균 None, 등급 None

In [11]:
# Q2: 사용자 정의 예외와 자모 분류
import logging
logging.basicConfig(level=logging.INFO)

# (a) 사용자 정의 예외
class InvalidJamoError(ValueError):
    """한글 자모가 아닌 문자가 입력됐을 때 발생하는 예외."""


# (b) classify_jamo 함수
def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 필요합니다")
    if len(c) != 1:
        raise ValueError(f"길이가 1인 문자열이 필요합니다: {c!r}")
    
    code = ord(c)
    if 0x3131 <= code <= 0x314E:
        return "자음"
    elif 0x314F <= code <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c!r}")


# (c) 입력 리스트 처리
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1인 문자열이 필요합니다: 'AB'
[TypeError] str 타입이 필요합니다
자음
모음
[ValueError] 길이가 1인 문자열이 필요합니다: ''


설명: InvalidJamoError를 ValueError의 자식으로 만드는 이유는, ValueError가 "자료형은 맞지만 값이 조건을 충족하지 못할 때" 쓰는 표준예외이다. 자모가 아닌 한 글자 문자열이 들어온 상황은 정확히 이 경우에 해당하므로, Exception보다 ValueError를 부모로 삼는 것이 더 부합하다. except 순서가 중요한 이유는, 부모 예외로 잡으면 자식 예외도 모두 잡히기 때문에, 자식 -> 부모 순으로 잡아야 각각 잡을 수 있다. 따라서 InvalidJamoError를 먼저 잡았다. 한글 자음, 모음의 유니코드의 코드 포인트와 비교하여 문제를 처리하였다. 
참조한 생성형 ai 링크: https://claude.ai/share/5a7eef58-2b1d-4963-916a-3e16709dba11

위 inputs에 대한 출력 결과 :
자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1인 문자열이 필요합니다: 'AB'
[TypeError] str 타입이 필요합니다
자음
모음
[ValueError] 길이가 1인 문자열이 필요합니다: ''